# llmbenchmark quickstart: Claude vs GPT on a small QA set

This notebook:
1. calls two models over a small, **versioned** dataset and **records every call** with `llmbenchmark`,
2. **evaluates** the answers with DeepEval, using Claude as the judge,
3. **compares** the two runs item by item.

**Before running:** start the backend from the repo root:
```bash
LLMBENCHMARK_SERVER_ALLOW_ANONYMOUS=1 .venv/bin/uvicorn server.app:create_app --factory --reload
```
and set `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` in the environment that launched Jupyter.

In [ ]:
# Local development: install the SDK from this repo (editable).
%pip install -q -e "..[eval]" anthropic openai

# From your private index instead:
# %pip install -q --index-url https://<artifactory>/api/pypi/<repo>/simple "llmbenchmark[eval]==0.1.*"

In [ ]:
import os

# DeepEval sends anonymous usage telemetry unless you opt out.
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

for key in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
    print(key, "set" if os.environ.get(key) else "MISSING")

## 1. Configure the SDK

In [ ]:
import llmbenchmark

llmbenchmark.init(
    endpoint="http://localhost:8000",   # or set LLMBENCHMARK_ENDPOINT
    api_key=None,                        # or set LLMBENCHMARK_API_KEY (your team's key)
    project="quickstart",
)
llmbenchmark.__version__

## 2. A versioned prompt and dataset

In [ ]:
prompt = llmbenchmark.Prompt("qa", system="Answer the question in one short sentence.",
                             template="{input}", version="v1")
dataset = llmbenchmark.Dataset("qa-sample", [
    {"id": "france",   "input": "What is the capital of France?",                              "expected_output": "Paris"},
    {"id": "austen",   "input": "Who wrote 'Pride and Prejudice'?",                            "expected_output": "Jane Austen"},
    {"id": "gold",     "input": "What is the chemical symbol for gold?",                       "expected_output": "Au"},
    {"id": "continents","input": "How many continents are there?",                             "expected_output": "Seven"},
    {"id": "mars",     "input": "What planet is known as the Red Planet?",                     "expected_output": "Mars"},
    {"id": "boiling",  "input": "What is the boiling point of water at sea level in Celsius?", "expected_output": "100 degrees Celsius"},
    {"id": "monalisa", "input": "Who painted the Mona Lisa?",                                  "expected_output": "Leonardo da Vinci"},
    {"id": "ocean",    "input": "What is the largest ocean on Earth?",                         "expected_output": "The Pacific Ocean"},
], version="v1")
print(prompt, dataset, sep="\n")

## 3. Call each model and record every call

In [ ]:
import anthropic, openai

claude = anthropic.Anthropic()
gpt = openai.OpenAI()

MODELS = {"claude": "claude-opus-5-5", "gpt": "gpt-5"}   # change gpt-5 to any chat model your account has
MAX_TOKENS = 1024


def ask_claude(item):
    return claude.messages.create(
        model=MODELS["claude"], max_tokens=MAX_TOKENS,
        system=prompt.system, messages=prompt.messages(item),
        # Refusal fallbacks: if a safety classifier declines, the API retries on
        # another model instead of returning a refusal. Claude API only.
        extra_headers={"anthropic-beta": "server-side-fallback-2026-07-01"},
        extra_body={"fallbacks": "default"},
    )


def ask_gpt(item):
    return gpt.chat.completions.create(model=MODELS["gpt"], messages=prompt.messages(item, include_system=True))


runs = {}
for label, ask in [("claude", ask_claude), ("gpt", ask_gpt)]:
    with llmbenchmark.start_run(f"qa-{label}", model=MODELS[label], generation_params={"max_tokens": MAX_TOKENS},
                                prompt=prompt, dataset=dataset) as run:
        for item in dataset:
            # track() times the call; model/output/tokens are read from t.response.
            # raise_errors=False: one failed call is recorded and the loop continues.
            with run.track(item=item, raise_errors=False) as t:
                t.response = ask(item)
    runs[label] = run
    print(run)

In [ ]:
import pandas as pd

records = pd.concat([r.to_dataframe() for r in runs.values()])
records[["model", "input", "output", "input_tokens", "output_tokens", "latency_ms", "error"]]

In [ ]:
# Cost/speed comparison straight from the recorded metrics.
records.groupby("model")[["input_tokens", "output_tokens", "latency_ms"]].agg(["mean", "sum"]).round(1)

## 4. Evaluate with DeepEval (Claude as the judge)

- `AnswerRelevancyMetric`: does the answer address the question?
- `correctness()`: a G-Eval metric comparing the answer with the expected answer.

Each metric × record is a judge call, so this costs tokens. Read some `reason`s
to check that you agree with the judge before trusting the scores. Each call to
`evaluate_run` is stored as one versioned **evaluation**.

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric
from llmbenchmark.eval import evaluate_run, judge, correctness

j = judge()   # ClaudeJudge("claude-sonnet-5-5"); judge("gpt-4.1") would use OpenAI instead

reports = {}
for label, run in runs.items():
    metrics = [AnswerRelevancyMetric(threshold=0.7, model=j), correctness(model=j)]
    reports[label] = evaluate_run(run, metrics=metrics)   # also uploads the evaluation to the backend

In [ ]:
pd.concat({label: r.summary() for label, r in reports.items()}, names=["run"]).round(3)

In [ ]:
scores = pd.concat([r.to_dataframe() for r in reports.values()])
scores.merge(records[["id", "model", "input", "output"]], left_on="record_id", right_on="id")[
    ["model", "input", "output", "metric", "score", "passed", "reason"]
]

## 5. Compare the two runs item by item

In [ ]:
diff = llmbenchmark.compare(base=reports["claude"], head=reports["gpt"])
print(diff.warnings or "like-for-like: same dataset version and scoring config")
display(diff.summary())
pd.DataFrame(diff.regressions)

## 6. Read it back from the backend

In [ ]:
import httpx

base = llmbenchmark.get_config().endpoint
for label, run in runs.items():
    stored = httpx.get(f"{base}/v1/runs/{run.id}/records").json()
    evals = httpx.get(f"{base}/v1/runs/{run.id}/evaluations").json()
    print(label, run.id, "records:", len(stored), "evaluations:", [e["status"] for e in evals])